# Day03：淘宝商品数据Pandas探索

**姓名：** 24012453

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。

## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。

## 任务0：环境与个人配置

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


STUDENT_NAME = "24012453"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到data/淘宝全品类全国数据.csv")


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"
OUTPUT_DIR = ROOT / "output" / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert STUDENT_NAME != "请填写姓名", "请先填写姓名"
print("姓名：", STUDENT_NAME)
print("数据：", DATA_PATH)
print("输出：", OUTPUT_DIR)

## 任务1：读取数据并完成初步观察

In [ ]:
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
print("字段名：", df.columns.tolist())
display(df.head())

In [ ]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")

**数据粒度：** 请用一句话说明一行代表什么。

> 一行代表一条淘宝商品记录，包含该商品的id、品类、价格、销量、省份等信息。

## 任务2：字段类型与缺失值

In [ ]:
# TODO 1：输出字段类型
print(df.dtypes)

# TODO 2：计算缺失数量并从高到低排序，变量名为missing_count
missing_count = df.isna().sum().sort_values(ascending=False)

# TODO 3：计算缺失率（百分比），变量名为missing_rate
missing_rate = (df.isna().mean() * 100).round(1).sort_values(ascending=False)

# TODO 4：展示结果
print("缺失数量：")
print(missing_count)
print("\n缺失率（%）：")
print(missing_rate)

In [ ]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")

请填写：

- 一个可以直接进行数值统计的字段及原因；
- 一个暂不适合直接进行精确数值统计的字段及原因。

> - **可以直接进行数值统计的字段：** `商品价格`，因为它是数值型（float64）字段，可以直接使用 mean、median、describe 等方法进行统计计算。
> - **暂不适合直接进行精确数值统计的字段：** `商品销量`，因为它包含"100+人付款""1万+人付款"等文本格式数据，属于 object 类型，无法直接进行数值运算，需要先做数据清洗和转换。

## 任务3：选择列与选择行

In [ ]:
# TODO 1：选择"商品价格"列，变量名为price_series
price_series = df["商品价格"]

# TODO 2：选择商品id、一级品类、商品价格、省份、商品销量五列
product_view = df[["商品id", "一级品类", "商品价格", "省份", "商品销量"]]

# TODO 3：分别使用loc和iloc取前5行局部数据
loc_view = df.loc[0:4, ["一级品类", "商品价格", "省份"]]
iloc_view = df.iloc[0:5, 0:4]

# TODO 4：展示结果
print("price_series 类型：", type(price_series))
print("product_view 形状：", product_view.shape)
print("\nloc_view（前5行）：")
display(loc_view)
print("\niloc_view（前5行）：")
display(iloc_view)

In [ ]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")

请解释`df["商品价格"]`与`df[["商品价格"]]`的区别。

> `df["商品价格"]` 返回的是一个 **Series**（一维数据结构），相当于提取单列数据，可以直接调用 Series 的方法如 `.mean()`、`.describe()` 等。
> `df[["商品价格"]]` 返回的是一个 **DataFrame**（二维数据结构），保留了表格结构，可以继续使用 DataFrame 的方法如 `.head()`、`.info()` 等。

## 任务4：条件筛选与排序

In [ ]:
# TODO 1：筛选广东商品，变量名为guangdong
guangdong = df[df["省份"] == "广东"]

# TODO 2：筛选广东且商品价格不低于1000元的商品
# 只保留商品id、一级品类、二级品类、商品价格、省份、商品销量
# 按商品价格从高到低排序，变量名为guangdong_high_price
condition = (df["省份"] == "广东") & (df["商品价格"] >= 1000)
guangdong_high_price = df.loc[condition, ["商品id", "一级品类", "二级品类", "商品价格", "省份", "商品销量"]]\
    .sort_values(by="商品价格", ascending=False)

# TODO 3：筛选浙江或江苏商品，变量名为zhejiang_or_jiangsu
zhejiang_or_jiangsu = df[(df["省份"] == "浙江") | (df["省份"] == "江苏")]

# TODO 4：展示广东高价商品前10行和浙江或江苏商品数
print("广东高价商品前10行：")
display(guangdong_high_price.head(10))
print("\n浙江或江苏商品数：", zhejiang_or_jiangsu.shape[0])

In [ ]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")

## 任务5：描述性统计与一级品类汇总

In [ ]:
# TODO 1：使用describe查看商品价格摘要，变量名为price_summary
price_summary = df["商品价格"].describe()

# TODO 2：按一级品类统计商品数、平均价格和中位价格
# 按平均价格从高到低排序，变量名为category_summary
category_summary = (
    df.groupby("一级品类")
      .agg(商品数=("商品id", "size"),
           平均价格=("商品价格", "mean"),
           中位价格=("商品价格", "median"))
      .sort_values("平均价格", ascending=False)
      .round(2)
)

# TODO 3：展示结果
print("商品价格描述性统计：")
print(price_summary)
print("\n一级品类汇总（按平均价格从高到低）：")
display(category_summary)

In [ ]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")

请写一条一级品类价格结论，并说明它不能代表什么。

> 从一级品类汇总结果来看，不同品类之间的平均标价差异明显，平均价格最高的品类与最低的品类差距较大，反映出不同商品类别的定价区间差异。
> **结论边界：** 上述统计基于商品标价而非实际成交金额，且仅反映数据集中各品类的商品数量分布，不代表市场整体情况或消费者实际购买行为。

## 挑战任务：省份—类别分析

In [ ]:
# TODO 1：选择两个省份
provinces = ["广东", "江苏"]

# TODO 2：筛选省份并统计商品数、平均价格和中位价格
province_summary = (
    df[df["省份"].isin(provinces)]
    .groupby("省份")
    .agg(商品数=("商品id", "size"),
         平均价格=("商品价格", "mean"),
         中位价格=("商品价格", "median"))
    .round(2)
)

# TODO 3：分别计算两个省份最常见的一级品类
top_categories_list = []
for prov in provinces:
    prov_df = df[df["省份"] == prov]
    top_cat = prov_df["一级品类"].value_counts().head(3)
    for cat, cnt in top_cat.items():
        top_categories_list.append({"省份": prov, "一级品类": cat, "商品数": cnt})
top_categories = pd.DataFrame(top_categories_list)

# TODO 4：展示结果
print("省份汇总统计：")
display(province_summary)
print("\n各省份最常见的一级品类（Top3）：")
display(top_categories)

In [ ]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")

## 输出成果

In [ ]:
outputs = {
    "category_summary.csv": category_summary.reset_index(),
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))

## 结论与边界

请至少完成两条结论，每条包含：数据范围、字段与方法、数据结论、结论边界。

### 结论1

> **数据范围：** 本分析基于淘宝全品类全国数据集，共25,000条商品记录。
> **字段与方法：** 使用`商品价格`字段，通过`describe()`方法获取描述性统计指标（均值、标准差、四分位数等）。
> **数据结论：** 商品标价的均值约为938.26元，中位数与均值存在一定差距，说明价格分布呈右偏态，存在部分高价商品拉高平均值。
> **结论边界：** `商品价格`是商品标价，不代表实际成交金额；统计结果仅反映数据集中25,000条记录的情况，不能推断为全平台或全品类商品的真实价格分布。

### 结论2

> **数据范围：** 基于数据集中广东和江苏两省的淘宝商品记录。
> **字段与方法：** 使用`省份`、`一级品类`、`商品价格`字段，通过分组聚合（groupby）统计各省份的商品数、平均价格和中位价格，并通过value_counts找出最常见品类。
> **数据结论：** 广东和江苏两省在商品数量和品类分布上存在差异，不同省份的优势品类不同，反映了区域消费和供给的差异性。
> **结论边界：** 数据仅包含淘宝平台部分商品，不代表全电商平台情况；商品标价不等于成交价；省份分类可能不完全反映实际发货地或商家所在地。

### GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] 检查点1～6全部通过；
- [ ] `output/day03_analysis/`包含两个CSV；
- [ ] 结论明确说明商品标价不代表实际成交金额；
- [ ] 已提交并推送到个人GitHub仓库。